In [83]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 15, 6

import openturns as ot
import openturns.viewer as otv
ot.Log.Show(ot.Log.NONE)

import functions_probabilistic as fp

In [84]:
# Constants
global gamma_w, gamma_s, d70_m, L, H, tan_theta
global eta, nu, g, d, rho_sub
global Lv_clay, Lv_sand, L_add

gamma_w = 1025 * 9.81 # unit weight of the water
gamma_s = 16500 # unit weight of the submerged particle

# d70 = 2.8e-4 # 70%-fractile of grain size distribution
d70_m = 2.08e-4 # Reference value of 70%-fractile of grain size dis-tribution

H = 5.3 # water level at the foreside of the dike
tan_theta = np.tan(37 * (np.pi/180)) # slope of the dike
# k = 7.52e-4 # hydraulic conductivity of the auqifer
# D = 6.0 # thickness of the aquifer
eta = 0.25 # Drag factor coefficient 
# m_p = 1 # Model factor piping
nu = 1.33e-6 # Kinematic viscosity 
g = 9.81 # Gravitational acceleration
# h_b = 0 # water level on the hinter side of the dike
d = 2.5 - 0.5 # impermeable clay layer at the sand boil exit point (adjusted)
rho_sub = 1.25 # factor of safety

# C_clay = 8.5
# C_sand = 6

L_add = 0

Lv_sand = 6
L = 45.4 + 9 # piping length (adjusted to new situation)
print('L = ', L)

L =  54.4


In [85]:
# Variables
# d70 = ot.LogNormal(2.8e-4, 0.12)
# k = ot.LogNormal(7.52e-4, 0.50)
# D = ot.LogNormal(6.0, 0.25)
m_p = ot.Normal(1.0, 0.12)
h_p = ot.Normal(-0.5, 0.1)
# x = (d70, k, D, m_p, h_p)
C_clay = ot.Normal(2.3, 0.1)
C_sand = ot.Normal(6.25, 0.1)
x = (m_p, h_p, C_sand, C_clay)
descriptions = ["model factor m_p",
             "phreatic level hinterland h_p",
             "coefficient C_sand",
             "coefficient C_clay"]
# descriptions = ["grain size d70",
#                 "hydraulic conductivity k",
#                 "thickness of the aquifer D",
#                 "model factor m_p",
#                 "phreatic level hinterland h_p"]

In [86]:
def LSF(x):
    m_p, h_p, C_sand, C_clay = x
    # F_R = (eta*((gamma_s/gamma_w)-1)*tan_theta)
    # F_S = ((d70_m / (((nu*k*L)/g)**(1/3))) * ((d70/d70_m)**0.4))
    # step_1 = ((D/L)**2.8)-1
    # step_2 = (0.28/(step_1)) + 0.04
    # F_G = (0.91*((D/L)**step_2))
    if Lv_sand + L_add > 11:
        print('Warning: Lv_sand + L_add > 11, check the values!')
    if Lv_sand + L_add > 6 and Lv_sand + L_add < 9.5:
        H_c = (((L/3) + (4 * Lv_sand)) / C_sand) + ((4*L_add) / C_clay)
    if Lv_sand + L_add > 9.5:
        H_c = (((L/3) + (4 * Lv_sand)) / C_sand) + (4*2.5 / C_clay) + (4*(L_add - 2.5) / C_sand)
    else:
        H_c = (((L/3) + (4 * Lv_sand)) / C_sand)
    # H_c = m_p * F_R * F_S * F_G * L / rho_sub
    Z = (m_p * H_c) - (H - h_p - (d*0.3))

    # print(L_add)
    return [Z]


In [87]:
LSF((1, -0.5, 6.25, 2.3))

[1.5413333333333332]

In [88]:
fp.input_OpenTurns(x, descriptions, LSF, 0)

In [89]:
result, x_star, u_star, pf_FORM, beta = fp.run_FORM_analysis()

The FORM analysis took 0.025 seconds
FORM result, pf = 0.0300
FORM result, beta = 1.881

The design point in the u space:  [-1.85693,-0.230287,0.19174,-9.94808e-05]
The design point in the x space:  [0.777169,-0.523029,6.26917,2.29999]


In [90]:
it = 0
maxit = 100
while pf_FORM > 1.4e-6 and it < maxit:
    L_add += 0.1
    it += 1
    fp.input_OpenTurns(x, descriptions, LSF, 0)
    result, x_star, u_star, pf_FORM, beta = fp.run_FORM_analysis(printing=False)
    print(f"Iteration {it}: pf_FORM = {pf_FORM}, Lv_add = {L_add}")
L = Lv_sand + L_add
print(f"Final L = {L}, pf_FORM = {pf_FORM}, beta = {beta}")



Iteration 1: pf_FORM = 0.02998923318992899, Lv_add = 0.1
Iteration 2: pf_FORM = 0.02998923318992899, Lv_add = 0.2
Iteration 3: pf_FORM = 0.02998923318992899, Lv_add = 0.30000000000000004
Iteration 4: pf_FORM = 0.02998923318992899, Lv_add = 0.4
Iteration 5: pf_FORM = 0.02998923318992899, Lv_add = 0.5
Iteration 6: pf_FORM = 0.02998923318992899, Lv_add = 0.6
Iteration 7: pf_FORM = 0.02998923318992899, Lv_add = 0.7
Iteration 8: pf_FORM = 0.02998923318992899, Lv_add = 0.7999999999999999
Iteration 9: pf_FORM = 0.02998923318992899, Lv_add = 0.8999999999999999
Iteration 10: pf_FORM = 0.02998923318992899, Lv_add = 0.9999999999999999
Iteration 11: pf_FORM = 0.02998923318992899, Lv_add = 1.0999999999999999
Iteration 12: pf_FORM = 0.02998923318992899, Lv_add = 1.2
Iteration 13: pf_FORM = 0.02998923318992899, Lv_add = 1.3
Iteration 14: pf_FORM = 0.02998923318992899, Lv_add = 1.4000000000000001
Iteration 15: pf_FORM = 0.02998923318992899, Lv_add = 1.5000000000000002
Iteration 16: pf_FORM = 0.0299892

In [91]:
fp.run_MonteCarloSimulation(100000)

The MCS took 0.703 seconds to evaluate 100000 samples.
pf for MCS:  4e-05


np.float64(4e-05)